# MLOps fundamentals example

## Libraries

In [1]:
import mlflow
import mlflow.sklearn
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.datasets import fetch_covtype
from sklearn.metrics import accuracy_score


## Importing Data

Dataset reference: [Forest cover type (fetch_covtype)](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.fetch_covtype.html).

In [2]:
print("Loading data...")

X, y = fetch_covtype(return_X_y=True)

## 
# The samples in this dataset correspond to 30×30m patches of forest in the US,
# collected for the task of predicting each patch’s cover type, i.e. the
# dominant species of tree. Details about the dataset can be found here: https://scikit-learn.org/stable/modules/generated/sklearn.datasets.fetch_covtype.html
##

# Fetch a representative subset for faster experimentation.
X, _, y, _ = train_test_split(
    X, y, train_size=50000, stratify=y, random_state=42
)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

print(f"Training data shape: X_train={X_train.shape}, y_train={y_train.shape}")
print(f"Testing data shape: X_test={X_test.shape}, y_test={y_test.shape}")

Loading data...
Training data shape: X_train=(40000, 54), y_train=(40000,)
Testing data shape: X_test=(10000, 54), y_test=(10000,)


## Modelling

### Manual implementation

In [ ]:
# Here is where you define your model, including parameters you can scan later
# through hyperparameter tuning process.
 
n_estimators = 200
max_depth = 20
min_samples_leaf = 2
random_state = 42

model = RandomForestClassifier(
        n_estimators=n_estimators,
        min_samples_leaf=min_samples_leaf,
        max_depth=max_depth,
        random_state=random_state,
        n_jobs=-1,
    )

In [ ]:
# Once the model is defined training and evaluation can be done

model.fit(X_train, y_train)
print("Model training complete.")

print("Evaluating model...")
y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(f"Model accuracy on test set: {accuracy:.4f}")


### Now with MLFlow

In [ ]:
with mlflow.start_run():
    n_estimators = 200
    max_depth = 20
    min_samples_leaf = 2
    random_state = 42

    model = RandomForestClassifier(
        n_estimators=n_estimators,
        min_samples_leaf=min_samples_leaf,
        max_depth=max_depth,
        random_state=random_state,
        n_jobs=-1,
    )
    model.fit(X_train, y_train)
    # print("Model training complete.")

    # print("Evaluating model...")
    y_pred = model.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)

    # The folllowing lines are the ones that will log the model and metrics to MLflow.
    # You can add more parameters and metrics as needed.
    mlflow.log_metric("accuracy", accuracy)
    mlflow.sklearn.log_model(
    model,
    name="model",
    skops_trusted_types=["sklearn.tree._tree.Tree"],
)


## Now screen several values for the model parameter and log the experiments in mlflow

In [5]:
from itertools import product

with mlflow.start_run(run_name="random_forest_search"):
    random_state = 42

    for n_estimators, max_depth, min_samples_leaf in product(
        [50, 100, 200, 300], [5, 10, 20], [1, 2, 3]
    ):
        with mlflow.start_run(
            run_name=f"rf_trees_{n_estimators}_depth_{max_depth}_leaf_{min_samples_leaf}",
            nested=True,
        ):
            model = RandomForestClassifier(
                n_estimators=n_estimators,
                max_depth=max_depth,
                min_samples_leaf=min_samples_leaf,
                random_state=random_state,
                n_jobs=-1,
            )
            model.fit(X_train, y_train)

            y_pred = model.predict(X_test)
            accuracy = accuracy_score(y_test, y_pred)

            mlflow.log_params({
                "n_estimators": n_estimators,
                "max_depth": max_depth,
                "min_samples_leaf": min_samples_leaf,
                "random_state": random_state,
            })
            mlflow.log_metric("accuracy", accuracy)
            mlflow.sklearn.log_model(
                model,
                name="model",
                skops_trusted_types=["sklearn.tree._tree.Tree"],
            )

2026/11/19 14:43:58 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
2026/11/19 14:43:59 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
2026/11/19 14:44:01 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
2026/11/19 14:44:03 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
2026/11/19 14:44:05 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
2026/11/19 14:44:06 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be adde